In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# first i will import some lybareis so maybe i will use them
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
!pip install catboost



df = pd.read_csv(os.path.join(path, csv_files[0]))
print("Dataset loaded successfully.")

In [ ]:
# Task 2: Write your code here:

df.head()

In [ ]:
# Task 3: Write your code here:

df.info()

In [ ]:


df.describe()# Task 4: Write your code here:

In [ ]:
# Task 1: Write your code here:
#i will check first for missing values
print("\nMissing Values (df.isnull().sum()):")
print(df.isnull().sum())
#then i will start handling
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

#i will Fill Numerical
for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Filled missing values in {col} with median: {median_val}")

#i will  Fill Categorical
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"Filled missing values in {col} with mode: {mode_val}")



In [ ]:
# Task 2: Write your code here:
#i will check first to see
print("Checking for duplicate rows...")
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:

    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in cat_cols:
    if col != 'target':
        df[col] = le.fit_transform(df[col].astype(str))

print("Categorical variables encoded.")

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
target_col = 'Target'
X = df.drop(columns=[target_col])
y = df[target_col]

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [ ]:
# Task 5: Write your code here:
print(f"Default: {df['Target'].sum()}")
print(f"No Default: {(df['Target'] == 0).sum()}")

In [ ]:
#plot the diffrence to see if balnced or not
import matplotlib.pyplot as plt
import seaborn as sns

counts = [5269, 14732]
labels = ['Default', 'No Default']

plt.figure(figsize=(6, 4))
sns.barplot(x=labels, y=counts, palette=['salmon', 'lightgreen'])

plt.title('Target Distribution (Class Imbalance)')
plt.ylabel('Count')
plt.xlabel('Target Class')

plt.show()

In [ ]:
# Task 1: Write your code here:


from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from catboost import CatBoostClassifier
import numpy as np
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []

print("Starting 5-Fold Stratified Cross-Validation with CatBoost...\n")

for fold, (train_index, val_index) in enumerate(skf.split(X_scaled, y)):
    # Create the split for this fold
    X_train, X_val = X_scaled.iloc[train_index], X_scaled.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]


    model = CatBoostClassifier(verbose=0, random_state=42)
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_val)

    score = f1_score(y_val, y_pred)
    f1_scores.append(score)

    print(f"Fold {fold+1} | F1-Score: {score:.4f}")

# 5. Print the averaged score
print(f"\nAverage F1-Score: {np.mean(f1_scores):.4f}")



In [ ]:
feature_importances = model.get_feature_importance()
feature_names = X.columns

# Create DataFrame
fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})
fi_df = fi_df.sort_values(by='Importance', ascending=False)


golden_feature = fi_df.iloc[0]['Feature']
print(f" The Golden is: {golden_feature}")


plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=fi_df.head(10), palette='magma')
plt.title('Top 10 Features (Golden Feature Search)')
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here:


print(f"Retraining using ONLY: {golden_feature}...\n")


X_gold = X_scaled[[golden_feature]]


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gold_acc_scores = []

for fold, (train_index, val_index) in enumerate(skf.split(X_gold, y)):
    X_train, X_val = X_gold.iloc[train_index], X_gold.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    model_gold = CatBoostClassifier(verbose=0, random_state=42)
    model_gold.fit(X_train, y_train)

    y_pred = model_gold.predict(X_val)
    gold_acc_scores.append(accuracy_score(y_val, y_pred))

full_model_acc = np.mean(acc_scores)
gold_model_acc = np.mean(gold_acc_scores)

print(f"Full Model Accuracy:      {full_model_acc:.4f}")
print(f"Golden Feature Accuracy:  {gold_model_acc:.4f}")
print(f"Difference:               {(full_model_acc - gold_model_acc):.4f}")

if (full_model_acc - gold_model_acc) < 0.05:
    print("\nConclusion: The Golden Feature is extremely powerful!")
else:
    print("\nConclusion: The Golden Feature is good, but other features are still needed.")